# Module 1 Exercise: MLP from scratch, then scale up

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nsteve2407/llm-transformers-course/blob/master/notebooks/01-dnn-refresher/exercise_starter.ipynb)

Module page: [Module 1: DNN Refresher](https://nsteve2407.github.io/llm-transformers-course/modules/01-dnn-refresher/)

**Part A**: implement a 2-layer MLP for MNIST using raw tensor ops only (manual forward + backward pass, no `.backward()`), and verify against autograd.

**Part B**: reimplement with `nn.Module`/autograd, train with SGD / SGD+momentum / Adam / AdamW vs. Adam+L2, and compare loss curves.

**Part C**: add Dropout + BatchNorm1d and demonstrate train vs. eval-mode behavior.

In [ ]:
import os
import torch
import torch.nn.functional as F

SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"
torch.manual_seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"SMOKE_TEST={SMOKE_TEST}, device={device}")

In [ ]:
if SMOKE_TEST:
    X_train = torch.rand(256, 784)
    y_train = torch.randint(0, 10, (256,))
    X_val = torch.rand(64, 784)
    y_val = torch.randint(0, 10, (64,))
else:
    from torchvision import datasets, transforms
    tfm = transforms.Compose([transforms.ToTensor(), transforms.Lambda(lambda x: x.view(-1))])
    train_ds = datasets.MNIST(root="./data", train=True, download=True, transform=tfm)
    val_ds = datasets.MNIST(root="./data", train=False, download=True, transform=tfm)
    X_train = torch.stack([train_ds[i][0] for i in range(len(train_ds))])
    y_train = torch.tensor([train_ds[i][1] for i in range(len(train_ds))])
    X_val = torch.stack([val_ds[i][0] for i in range(min(2000, len(val_ds)))])
    y_val = torch.tensor([val_ds[i][1] for i in range(min(2000, len(val_ds)))])

print(X_train.shape, y_train.shape)

## Part A: manual forward + backward pass, verified against autograd

In [ ]:
def init_params(d_in=784, d_hidden=128, d_out=10, seed=0):
    g = torch.Generator().manual_seed(seed)
    W1 = torch.randn(d_in, d_hidden, generator=g) * (2.0 / d_in) ** 0.5
    b1 = torch.zeros(d_hidden)
    W2 = torch.randn(d_hidden, d_out, generator=g) * (2.0 / d_hidden) ** 0.5
    b2 = torch.zeros(d_out)
    return [W1, b1, W2, b2]


def manual_forward(params, X):
    # TODO: compute z1 = X @ W1 + b1, a1 = relu(z1), logits = a1 @ W2 + b2
    # Return logits and a cache of (X, z1, a1) needed for the backward pass.
    raise NotImplementedError("TODO: implement manual_forward")


def manual_backward(params, cache, logits, y):
    # TODO: derive dlogits from softmax(logits) - one_hot(y), divided by batch size,
    # then backprop through fc2, ReLU, and fc1 to get dW1, db1, dW2, db2.
    raise NotImplementedError("TODO: implement manual_backward")


def cross_entropy_loss(logits, y):
    return F.cross_entropy(logits, y)

In [ ]:
# Gradient check: compare manual gradients to autograd on a small batch
params = init_params()
X_batch, y_batch = X_train[:32], y_train[:32]

autograd_params = [p.clone().requires_grad_(True) for p in params]
logits_ag, _ = manual_forward(autograd_params, X_batch)
loss_ag = cross_entropy_loss(logits_ag, y_batch)
loss_ag.backward()
autograd_grads = [p.grad for p in autograd_params]

logits_manual, cache = manual_forward(params, X_batch)
manual_grads = manual_backward(params, cache, logits_manual, y_batch)

max_diffs = [(a - m).abs().max().item() for a, m in zip(autograd_grads, manual_grads)]
print("Max abs diff per param (W1, b1, W2, b2):", max_diffs)
assert all(d < 1e-5 for d in max_diffs), "Manual and autograd gradients diverge!"
print("Manual backprop matches autograd.")

In [ ]:
# Train the manual-backprop MLP with plain SGD
params = init_params()
lr = 0.5
epochs = 1 if SMOKE_TEST else 5
batch_size = 64

for epoch in range(epochs):
    perm = torch.randperm(X_train.shape[0])
    total_loss = 0.0
    for i in range(0, X_train.shape[0], batch_size):
        idx = perm[i : i + batch_size]
        Xb, yb = X_train[idx], y_train[idx]
        logits, cache = manual_forward(params, Xb)
        loss = cross_entropy_loss(logits, yb)
        grads = manual_backward(params, cache, logits, yb)
        with torch.no_grad():
            for p, g in zip(params, grads):
                p -= lr * g
        total_loss += loss.item() * Xb.shape[0]
    val_logits, _ = manual_forward(params, X_val)
    val_acc = (val_logits.argmax(dim=1) == y_val).float().mean().item()
    print(f"epoch {epoch}: train_loss={total_loss / X_train.shape[0]:.4f} val_acc={val_acc:.4f}")

## Part B: autograd + optimizer comparison (SGD, SGD+momentum, Adam, AdamW vs. Adam+L2)

In [ ]:
import torch.nn as nn


class MLP(nn.Module):
    def __init__(self, d_in=784, d_hidden=128, d_out=10):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        # TODO: apply fc1, then relu, then fc2, and return the result.
        raise NotImplementedError("TODO: implement the two-layer forward pass")


def make_optimizer(name, model):
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=0.1)
    if name == "sgd_momentum":
        return torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9)
    if name == "adam":
        return torch.optim.Adam(model.parameters(), lr=1e-3)
    if name == "adam_l2":
        return torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.01)
    if name == "adamw":
        return torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    raise ValueError(name)


def train_variant(name, epochs):
    torch.manual_seed(0)
    model = MLP().to(device)
    opt = make_optimizer(name, model)
    losses = []
    for epoch in range(epochs):
        perm = torch.randperm(X_train.shape[0])
        total_loss = 0.0
        for i in range(0, X_train.shape[0], batch_size):
            idx = perm[i : i + batch_size]
            Xb, yb = X_train[idx].to(device), y_train[idx].to(device)
            opt.zero_grad()
            loss = F.cross_entropy(model(Xb), yb)
            loss.backward()
            opt.step()
            total_loss += loss.item() * Xb.shape[0]
        losses.append(total_loss / X_train.shape[0])
    return losses


epochs_b = 1 if SMOKE_TEST else 5
curves = {name: train_variant(name, epochs_b) for name in ["sgd", "sgd_momentum", "adam", "adam_l2", "adamw"]}
for name, losses in curves.items():
    print(name, [round(l, 4) for l in losses])

In [ ]:
import matplotlib.pyplot as plt

for name, losses in curves.items():
    plt.plot(losses, label=name)
plt.xlabel("epoch")
plt.ylabel("train loss")
plt.legend()
plt.title("Optimizer comparison, incl. Adam+L2 vs AdamW")
plt.show()

## Part C: Dropout + BatchNorm1d, train vs. eval mode

In [ ]:
class RegularizedMLP(nn.Module):
    def __init__(self, d_in=784, d_hidden=128, d_out=10, p=0.5):
        super().__init__()
        self.fc1 = nn.Linear(d_in, d_hidden)
        self.bn1 = nn.BatchNorm1d(d_hidden)
        self.dropout = nn.Dropout(p)
        self.fc2 = nn.Linear(d_hidden, d_out)

    def forward(self, x):
        # TODO: apply fc1 -> bn1 -> relu -> dropout -> fc2, and return the result.
        raise NotImplementedError("TODO: implement fc1 -> bn1 -> relu -> dropout -> fc2")


reg_model = RegularizedMLP().to(device)
x_probe = X_train[:8].to(device)

reg_model.train()
out_train_1 = reg_model(x_probe)
out_train_2 = reg_model(x_probe)
print("Train mode is stochastic (outputs differ):", not torch.allclose(out_train_1, out_train_2))

reg_model.eval()
out_eval_1 = reg_model(x_probe)
out_eval_2 = reg_model(x_probe)
print("Eval mode is deterministic (outputs match):", torch.allclose(out_eval_1, out_eval_2))